**Original Folder:** SUICIDE_WATCH-DATABASE

# User Info

In [2]:
import pandas as pd
import json
import requests
import time

## Compiles a List of Users from r/SuicideWatch

In [4]:
records = []
filepath = r'D:\Reddit\ZStandard\SuicideWatch_submissions'
with open(filepath, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        record = json.loads(line)
        records.append(record)
df = pd.DataFrame(records)
# Remove 'deleted' user
df = df[df['author'] != '[deleted]']
# Remove 'AutoModerator' user
df = df[df['author'] != 'AutoModerator']
userArray = df['author'].unique()
userArrayTest = df['author'].unique()[:10]
del df
## API Setup

CLIENT_ID = 'ee5DffSIb70G-g7EOVuG2A'
CLIENT_SECRET = 'NRBMN3dQLyygyRHsknNa69eTWrxXpw'
auth = requests.auth.HTTPBasicAuth(CLIENT_ID, CLIENT_SECRET)
with open(r"D:\Reddit\pw.txt", 'r') as f:
    pw = f.read()
data = {
    'grant_type': 'password',
    'username': 'Progzly',
    'password': pw
}
headers = {'User-Agent': 'MyAPI/0.0.1'}
res = requests.post('https://www.reddit.com/api/v1/access_token',
                    auth=auth, data=data, headers=headers)
TOKEN_DATA = res.json()
TOKEN = TOKEN_DATA['access_token']
# headers['Authorization'] = f'bearer {TOKEN}'
headers = {**headers, **{'Authorization': f'bearer {TOKEN}'}}
headers
# requests.get('https://oauth.reddit.com/api/v1/me', headers={'User-Agent': 'MyAPI/0.0.1'}).json()
requests.get('https://oauth.reddit.com/api/v1/me', headers=headers)
#Fetching Data
# res = requests.get('https://oauth.reddit.com/r/SuicideWatch/new', 
#                     headers=headers, params={'limit': 101})
## Get User Information
def getInfo(user):
    url = f'https://oauth.reddit.com/user/{user}/about'
    res = requests.get(url, headers=headers)
    if res.status_code == 200:
        return res.json()
    else:
        print(f"Error fetching data for user {user}: {res.status_code}")
        return {res.status_code}
### Recursive Information Fetch


offset = 3
for i in range(1): #10-1-offset
    print(f"Processing batch {i+offset}...")
    userArrayIter = userArray[29470*(i+offset):29470*(i+1+offset)]
    print(len(userArrayIter))
    userData = []
    nullArray = []
    for user in userArrayIter:
        failed_attempts = 0
        userInfo = getInfo(user)
        while userInfo == {429}:
            failed_attempts += 1
            if failed_attempts > 6:
                print(f"Rate limit exceeded for {user}. Skipping...")
                break
            print(f"Rate limit exceeded for {user} (Fail #{failed_attempts}). Retrying in 60 seconds...)")
            time.sleep(60)
            userInfo = getInfo(user)
        if userInfo and 'data' in userInfo:
            user_features = {
                'author': user,
                'cake_day_utc': userInfo['data'].get('created_utc', None),
                'karma': userInfo['data'].get('total_karma', None),
                'post_karma': userInfo['data'].get('link_karma', None),
                'comment_karma': userInfo['data'].get('comment_karma', None),
                'over_18': userInfo['data'].get('over_18', None)
            }
            userData.append(user_features)
        if userInfo == {404}:
            nullArray.append(user)
    userData = pd.DataFrame(userData, columns=['author', 'cake_day_utc', 'karma', 'post_karma', 'comment_karma', 'over_18'])
    nullArray = pd.DataFrame(nullArray, columns=['author'])
    userData.to_csv(f'D:/Reddit/ZStandard/userData{i+offset}.csv', mode='a', index=False, header=False)
    nullArray.to_csv(f'D:/Reddit/ZStandard/nullArray{i+offset}.csv', mode='a', index=False, header=False)
    print(f"Batch {i+offset} processed and saved.")
    time.sleep(60)
#

Processing batch 3...
29470
Error fetching data for user expandingxo: 404
Error fetching data for user Rhodelius: 404
Error fetching data for user lilypond7: 404
Error fetching data for user tjp1994: 404
Error fetching data for user moldymiso: 404
Error fetching data for user swimmingidiot: 404
Error fetching data for user cmdalek: 404
Error fetching data for user saracasticgoddess26: 404
Error fetching data for user budderbudder731: 404
Error fetching data for user nainamarbusataknev: 404
Error fetching data for user AngelaE_R: 404
Error fetching data for user ShatterMeStill: 404
Error fetching data for user Xhanon: 404
Error fetching data for user Schauma7: 404
Error fetching data for user vanillaawesome: 404
Error fetching data for user Skaardja: 404
Error fetching data for user Potato-Bill: 404
Error fetching data for user gautarinn1: 404
Error fetching data for user kimhelena123: 404
Error fetching data for user maybecomplete: 404
Error fetching data for user Raiichu_LoL: 404
Erro

In [ ]:
len(nullArray)
#169m for 10605

In [ ]:
# Save Null Array to CSV
nullArray = pd.DataFrame(nullArray)
nullArray.to_csv('D:/Reddit/ZStandard/NullArray0.csv', index=False, header=False)

In [ ]:
# Count Rows in .csv
test = pd.read_csv(r'D:\Reddit\ZStandard\userData0.csv', header=None)
test.columns = ['author', 'cake_day_utc', 'karma', 'post_karma', 'comment_karma', 'over_18']
len(test)


In [ ]:
userData['cake_day_dt'] = pd.to_datetime(userData['cake_day_utc'], unit='s')
userData